In [11]:
import warnings
warnings.filterwarnings('ignore')

import os
import cv2
import numpy as np
from PIL import Image

import torch
import torchvision
import torch.backends.cudnn as cudnn
import torchvision.transforms as transforms

from additional_files.post_processing.inference import get_final_preds
from additional_files.post_processing.transforms import get_affine_transform

from additional_files.models.pose_hr_net import get_pose_net

from additional_files.config.default_configuration import _C as cfg
from additional_files.config.default_configuration import update_config, COCO_INSTANCE_CATEGORY_NAMES, joints, activity_classes

In [12]:
from centroid_direction import CentroidTracker, TrackableObject

import os
from timeit import time
import warnings
import sys
import cv2
import numpy as np
from PIL import Image
from yolo import YOLO

from deep_sort import preprocessing
from deep_sort import nn_matching
from deep_sort.detection import Detection
from deep_sort.tracker import Tracker
from tools import generate_detections as gdet
from deep_sort.detection import Detection as ddet

yolo = YOLO()

trackableObjects = {}
ct = CentroidTracker(maxDisappeared=40, maxDistance=50)
# DeepSort Parameters Definition
max_cosine_distance = 0.3
nn_budget = None
nms_max_overlap = 1.0

peopleOut = 0
peopleIn = 0

# DeepSort Loading
model_filename = 'model_data/mars-small128.pb'
encoder = gdet.create_box_encoder(model_filename,batch_size=1)

metric = nn_matching.NearestNeighborDistanceMetric("cosine", max_cosine_distance, nn_budget)
tracker = Tracker(metric)

model_data/yolo.h5 model, anchors, and classes loaded.


In [13]:
def get_person_detection_boxes(person_model, img, threshold=0.5):
    pil_image = Image.fromarray(img)
    transform = transforms.Compose([transforms.ToTensor()])
    transformed_img = transform(pil_image)
    transformed_img = transformed_img.cuda()

    with torch.no_grad():
        pred = person_model([transformed_img])  
        pred_classes = [COCO_INSTANCE_CATEGORY_NAMES[i]
                        for i in list(pred[0]['labels'].cpu().numpy())]  
        pred_boxes = [[(i[0], i[1]), (i[2], i[3])]
                      for i in list(pred[0]['boxes'].cpu().numpy())]  
        pred_score = list(pred[0]['scores'].cpu().numpy())
        if not pred_score:
            return []
        
        pred_t = [pred_score.index(x) for x in pred_score if x > threshold][-1]
        pred_boxes = pred_boxes[:pred_t+1]
        pred_classes = pred_classes[:pred_t+1]
    
        person_boxes = []
        for idx, box in enumerate(pred_boxes):
            if pred_classes[idx] == 'person':
                person_boxes.append(box)

        return person_boxes

In [14]:
def get_pose_estimation_prediction(pose_model, image, center, scale):

    rotation = 0
    trans = get_affine_transform(center, scale, rotation, cfg.MODEL.IMAGE_SIZE)
    model_input = cv2.warpAffine(image, trans, (int(cfg.MODEL.IMAGE_SIZE[0]), int(cfg.MODEL.IMAGE_SIZE[1])),
        flags=cv2.INTER_LINEAR)
    
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])

    model_input = transform(model_input).unsqueeze(0)

    with torch.no_grad():
        output = pose_model(model_input)
        preds, _ = get_final_preds(
            cfg,
            output.clone().cpu().numpy(),
            np.asarray([center]),
            np.asarray([scale]))

        return preds

In [5]:
def box_to_center_scale(box, model_image_width, model_image_height):

    center = np.zeros((2), dtype=np.float32)

    bottom_left_corner = box[0]
    top_right_corner = box[1]
    box_width = top_right_corner[0]-bottom_left_corner[0]
    box_height = top_right_corner[1]-bottom_left_corner[1]
    bottom_left_x = bottom_left_corner[0]
    bottom_left_y = bottom_left_corner[1]
    center[0] = bottom_left_x + box_width * 0.5
    center[1] = bottom_left_y + box_height * 0.5

    aspect_ratio = model_image_width * 1.0 / model_image_height
    pixel_std = 200

    if box_width > aspect_ratio * box_height:
        box_height = box_width * 1.0 / aspect_ratio
    elif box_width < aspect_ratio * box_height:
        box_width = box_height * aspect_ratio
    scale = np.array([box_width * 1.0 / pixel_std, box_height * 1.0 / pixel_std], dtype=np.float32)
    if center[0] != -1:
        scale = scale * 1.25

    return center, scale

In [15]:
pose_dir = 'outputs/poses/'
box_dir = 'outputs/boxes/'
images_dir = 'outputs/poses/'

In [16]:
class Args:
  cfg = 'additional_files/config/inference-config.yaml'
  videoFile = 'data/train_1.mp4'
  outputDir = 'outputs/'
  inferenceFps = 10
  writeBoxFrames = True
  MODEL_FILE = 'additional_files/models/pytorch/pose_coco/pose_hrnet_w32_384x288.pth'
  modelDir = ''
  logDir = ''
  dataDir = ''

# cudnn related setting
cudnn.benchmark = cfg.CUDNN.BENCHMARK
torch.backends.cudnn.deterministic = cfg.CUDNN.DETERMINISTIC
torch.backends.cudnn.enabled = cfg.CUDNN.ENABLED

args=Args()
update_config(cfg, args)

box_model = torchvision.models.detection.fasterrcnn_resnet50_fpn(pretrained=True)
box_model.cuda()
box_model.eval()

pose_model = get_pose_net(cfg, is_train=False)
pose_model.load_state_dict(torch.load(args.MODEL_FILE), strict=False)
pose_model = torch.nn.DataParallel(pose_model, device_ids=cfg.GPUS)
pose_model.eval()

print('Models initialized.')

Models initialized.


In [19]:
from additional_files.sort import Sort

mot_tracker = Sort()

# single image
def run(image_bgr):
     image = image_bgr[:, :, [2, 1, 0]]

     # object detection box
     pred_boxes = get_person_detection_boxes(box_model, image, threshold=0.8)


     # circus for kalman filter
     pred_boxes_new = []
     for value_1 in pred_boxes:
         my_1 = []
         for value_2 in value_1:
            my_1.extend(value_2)
         pred_boxes_new.append(my_1)
     pred_boxes_new = np.array(pred_boxes_new)

     track_bbs_ids = mot_tracker.update(pred_boxes_new)

     image_bgr_box = image_bgr.copy()
     if pred_boxes:
         for box, id in zip(pred_boxes, track_bbs_ids):
             cv2.rectangle(image_bgr_box, box[0], box[1], color=(0, 255, 0), thickness=1)

             # pose estimation
             center, scale = box_to_center_scale(box, cfg.MODEL.IMAGE_SIZE[0], cfg.MODEL.IMAGE_SIZE[1])
             image_pose = image.copy() if cfg.DATASET.COLOR_RGB else image_bgr.copy()
             pose_preds = get_pose_estimation_prediction(pose_model, image_pose, center, scale)

             for _, mat in enumerate(pose_preds[0]):
                 x_coord, y_coord = int(mat[0]), int(mat[1])
                 cv2.circle(image_bgr, (x_coord, y_coord), 4, (255, 0, 0), 1)

             for i, joint in enumerate(joints['coco']['skeleton']):
                 pt1, pt2 = pose_preds[0][joint]
                 cv2.line(image_bgr, (int(pt1[0]), int(pt1[1])), (int(pt2[0]), int(pt2[1])), (80, 80, 255), 2)

             x,y,w,h = cv2.boundingRect(pose_preds[0])
             # cv2.rectangle(image_bgr, (x-10,y-10), (x+w+10,y+h+10), (80, 80, 255), thickness=1)
             cv2.rectangle(image_bgr, box[0], box[1], color=(180, 180, 0), thickness=2)
             cv2.putText(image_bgr, str(id[4]), (int(box[0][0]), int(box[0][1])-20), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 2)
     cv2.imwrite('single_image.png', image_bgr)


ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
import time

image_bgr = cv2.imread('DB175766.jpg')

start_time = time.time()
run(image_bgr)
stop_time = time.time()
print('elapsed time: {}s'.format(stop_time-start_time))

In [18]:
import cv2

vidcap = cv2.VideoCapture('data/The World Trade Center Station.mp4')
success, image_bgr = vidcap.read()

count = 0
W = None
H = None
max_id = 0
while success:

    if W is None or H is None:
        (H, W) = image_bgr.shape[:2]

    image = image_bgr[:, :, [2, 1, 0]]
    count_str = str(count).zfill(32)

    # object detection box
    pred_boxes = get_person_detection_boxes(box_model, image, threshold=0.8)

    image_kalman = Image.fromarray(image_bgr[...,::-1]) #bgr to rgb
    boxs = yolo.detect_image(image_kalman)

    if not pred_boxes:
        success, image_bgr = vidcap.read()
        count += 1
        continue

    # # circus for kalman filter
    # pred_boxes_new = []
    # for value_1 in pred_boxes:
    #     my_1 = []
    #     for value_2 in value_1:
    #        my_1.extend(value_2)
    #     pred_boxes_new.append(my_1)
    # pred_boxes_new = np.array(pred_boxes_new)

    for box in pred_boxes:
        # pose estimation
        center, scale = box_to_center_scale(box, cfg.MODEL.IMAGE_SIZE[0], cfg.MODEL.IMAGE_SIZE[1])
        image_pose = image.copy() if cfg.DATASET.COLOR_RGB else image_bgr.copy()
        pose_preds = get_pose_estimation_prediction(pose_model, image_pose, center, scale)

        for _, mat in enumerate(pose_preds[0]):
            x_coord, y_coord = int(mat[0]), int(mat[1])
            cv2.circle(image_bgr, (x_coord, y_coord), 4, (255, 0, 0), 1)

        for i, joint in enumerate(joints['coco']['skeleton']):
            pt1, pt2 = pose_preds[0][joint]
            cv2.line(image_bgr, (int(pt1[0]), int(pt1[1])), (int(pt2[0]), int(pt2[1])), (80, 80, 255), 1)

        x,y,w,h = cv2.boundingRect(pose_preds[0])
        # cv2.rectangle(image_bgr, (x-10,y-10), (x+w+10,y+h+10), (80, 80, 255), thickness=1)
        cv2.rectangle(image_bgr, box[0], box[1], color=(180, 180, 0), thickness=1)
        # cv2.putText(image_bgr, str(id[4]), (int(box[0][0]), int(box[0][1])-20), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 2)

    features = encoder(image_bgr, boxs)
    detections = [Detection(bbox, 1.0, feature) for bbox, feature in zip(boxs, features)]

    # Run non-maxima suppression.
    boxes = np.array([d.tlwh for d in detections])
    scores = np.array([d.confidence for d in detections])
    indices = preprocessing.non_max_suppression(boxes, nms_max_overlap, scores)
    detections = [detections[i] for i in indices]

    # Call the tracker
    tracker.predict()
    tracker.update(detections, H)
    scaler = H/1.5
    for track in tracker.tracks:
        if not track.is_confirmed() or track.time_since_update > 1:
            continue
        bbox = track.to_tlbr()

        c1 = (int(bbox[0]) + int(bbox[2]))/2
        c2 = (int(bbox[1]) + int(bbox[3]))/2
        centerPoint = (int(c1), int(c2))
        cv2.putText(image_bgr, str(track.track_id),centerPoint, 0, 5e-3 * 100, (180, 180, 0), 1)
        cv2.circle(image_bgr, centerPoint, 4, (80, 80, 255), 1)

        if track.stateOutMetro == 1 and (scaler +50 - (int(bbox[3]) + int(bbox[1]))/2 < 0) and track.noConsider == False:
            peopleOut += 1
            track.stateOutMetro = 0
            track.noConsider = True
            cv2.line(image_bgr, (0, int(scaler) +50), (W, int(scaler) +50), (0, 0, 0), 1)

        if track.stateOutMetro == 0 and (scaler +50 - (int(bbox[3]) + int(bbox[1]))/2 >= 0) and track.noConsider == False :
            peopleIn += 1
            track.stateOutMetro = 1
            track.noConsider = True
            cv2.line(image_bgr, (0, int(scaler) +50), (W, int(scaler) +50), (0, 0, 0), 1)

        if track.track_id > max_id:
            max_id = track.track_id

    cv2.line(image_bgr, (0, int(scaler) +50), (W, int(scaler) + 50), (0, 0, 255), 1)

    info = [
        ("Total People", max_id),
        ("People Count In", peopleIn),
        ("People Count Out", peopleOut)
    ]

    for (i, (k, v)) in enumerate(info):
        text = "{}: {}".format(k, v)
        cv2.putText(image_bgr, text, (20, H - ((i * 30) + 30)),
        cv2.FONT_HERSHEY_SIMPLEX, 1.1, (80, 80, 255), 1)

    cv2.imwrite(pose_dir+'pose%s.jpg' % count_str, image_bgr)

    success, image_bgr = vidcap.read()
    count += 1

KeyboardInterrupt: 

In [10]:
import imageio

images = []
count = 0
for file_name in os.listdir(pose_dir):
    # if count % 2 == 0:
    # if file_name.endswith('.jpg'):
    file_path = os.path.join(pose_dir, file_name)
    images.append(imageio.imread(file_path))
    count += 1
imageio.mimsave('outputs/together.gif', images)

In [ ]:
pose_dir = 'outputs/exp4'

import imageio

images = []
count = 0
for file_name in os.listdir(pose_dir):
    images.append(imageio.imread(file_name))